# Lab 01 — Playground

**Purpose:** Your sandbox for tweaking Lab 01 parameters and seeing what happens. The `src/` scripts stay untouched; you experiment here.

**Rule of thumb:** if an experiment produces a finding worth keeping, promote it — either save the results as a markdown cell in this notebook, or copy the winning config back into `src/` and note *why* in the commit message.

---

## What you can play with

1. **Chunk size / overlap** — different values, then re-embed and re-evaluate.
2. **Embedding model** — swap `all-MiniLM-L6-v2` for `all-mpnet-base-v2` (better quality, slower) or `BAAI/bge-small-en-v1.5` (competitive, small).
3. **Your own queries** — add queries beyond the 15 labeled ones and see what comes back.
4. **Your own corpus text** — add new paragraphs and check they surface for relevant queries.
5. **Hybrid weights** — change how BM25 and vector scores are fused.

## 0. Setup — make sure this notebook can find the `src/` code

In [ ]:
import sys
from pathlib import Path

# This notebook lives at labs/01-embeddings-semantic-search/experiments/
# The src/ code is one directory up.
LAB_ROOT = Path.cwd().parent
SRC_DIR = LAB_ROOT / "src"
DATA_DIR = LAB_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"LAB_ROOT = {LAB_ROOT}")
print(f"SRC_DIR  = {SRC_DIR}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"Corpus exists?  {(DATA_DIR / 'corpus.jsonl').exists()}")
print(f"Vectors exist?  {(DATA_DIR / 'vectors.npy').exists()}")

## 1. Import the lab's building blocks

The scripts in `src/` were written as *modules* — most of the useful bits are pure functions you can import. This is the ideal shape for lab code: scripts you can run from a terminal *and* functions you can call from a notebook.

> **Concept — why this matters (Staff-engineer lens):** most bootcamp code is "script-only" — logic tangled with `argparse` and file I/O. Production code separates *pure functions* (transform inputs to outputs, no I/O) from *drivers* (read files, call functions, write files). This is [functional core, imperative shell](https://www.destroyallsoftware.com/screencasts/catalog/functional-core-imperative-shell). It's what makes code testable, reusable, and easy to reason about. Interviewers notice.


In [ ]:
# Note: import names start with a digit so we use importlib.
import importlib

chunker_mod = importlib.import_module("01_ingest_chunk")
chunk_text = chunker_mod.chunk_text
content_hash = chunker_mod.content_hash

# quick smoke test
sample = "Data engineering is the practice of designing and operating systems that move and transform data at scale. " * 4
list(chunk_text(sample, size=32, overlap=8))[:2]

## 2. Experiment #1 — Chunk size sweep

**Question:** how does chunk size affect the number of chunks and average chunk length?

**Prediction:** smaller chunks → more chunks per doc, less context per chunk. Larger chunks → fewer chunks but each covers more concepts (embedding gets "averaged" and may lose sharpness on narrow queries).


In [ ]:
import json
import statistics

# Load the corpus (list of docs — NOT chunks)
docs = []
with (DATA_DIR / "corpus.jsonl").open() as f:
    for line in f:
        line = line.strip()
        if line:
            docs.append(json.loads(line))

print(f"Loaded {len(docs)} docs")

configs = [
    (16, 4),
    (32, 8),   # current default
    (64, 16),
    (128, 0),
    (200, 0),  # "no chunking" for this corpus — most docs fit in one chunk
]

results = []
for size, overlap in configs:
    chunks = []
    for doc in docs:
        for c in chunk_text(doc["text"], size, overlap):
            chunks.append(c)
    lens = [len(c.split()) for c in chunks]
    results.append({
        "size": size,
        "overlap": overlap,
        "n_chunks": len(chunks),
        "chunks_per_doc": round(len(chunks) / len(docs), 2),
        "avg_chunk_len": round(statistics.mean(lens), 1) if lens else 0,
        "max_chunk_len": max(lens) if lens else 0,
    })

for r in results:
    print(r)

**What to look for in the numbers above:**
- At `size=200`, does chunks-per-doc drop toward 1.0? (Most docs fit in a single chunk.)
- At `size=16`, does chunks-per-doc go up sharply? (Docs get split into many pieces.)
- Does average chunk length always equal `size`, or does it max out at doc length?

**Now run the actual embed + eval on your favorite config.** Pick one row above, edit `CHUNK_SIZE_TOKENS` and `CHUNK_OVERLAP_TOKENS` at the top of `src/01_ingest_chunk.py`, then in the cell below run the whole pipeline.

In [ ]:
# Run the full pipeline for the current config in src/.
# (Alternative: use subprocess.run so you can capture stdout.)
import subprocess

def run(script):
    r = subprocess.run(
        ["python", str(SRC_DIR / script)],
        capture_output=True, text=True, cwd=LAB_ROOT,
    )
    print(f"===== {script} =====")
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r

# Uncomment to run — takes ~10-30 sec end-to-end depending on model.
# run("01_ingest_chunk.py")
# run("02_embed_index.py")
# run("04_evaluate.py")

## 3. Experiment #2 — Search with your own queries

Use the existing index (`vectors.faiss` + `meta.parquet`) and search it with queries *you* care about.

In [ ]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

index = faiss.read_index(str(DATA_DIR / "vectors.faiss"))
meta = pd.read_parquet(DATA_DIR / "meta.parquet")
model = SentenceTransformer(MODEL_NAME)

def search(query: str, k: int = 5) -> pd.DataFrame:
    q_vec = model.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(q_vec.astype("float32"), k)
    hits = meta.iloc[idxs[0]].copy()
    hits["score"] = scores[0]
    cols = [c for c in ["score", "topic", "doc_id", "text"] if c in hits.columns]
    return hits[cols].reset_index(drop=True)

# ---- Try YOUR queries here ----
your_queries = [
    "how does spark decide the number of shuffle partitions",
    "what is a slowly changing dimension",
    "kafka vs kinesis for streaming",
]

for q in your_queries:
    print(f"\n=== Q: {q} ===")
    display(search(q, k=3))

**Interpretation guide:**
- `score` on a normalized-cosine setup ranges roughly 0.0-1.0. Above ~0.6 is a strong match, 0.3-0.5 is loose relevance, under 0.3 is noise.
- If a query returns *nothing* related, that's a legit finding — your corpus doesn't cover that topic. Add a paragraph to `data/corpus.jsonl`, re-run scripts 01 and 02, and try again.
- If a query returns *the wrong* topic, that's a retrieval failure worth investigating. Is the wording similar but the meaning different? That's a classic embedding-similarity-vs-semantic-relevance gap.

## 4. Experiment #3 — Try a stronger embedding model

MiniLM is fast (384 dims) but not the strongest. Try `all-mpnet-base-v2` (768 dims, better quality, ~3x slower).

**Interview lens:** the choice between MiniLM and MPNet is a real trade-off you'd defend in a system design interview. MiniLM at scale: cheaper embeds, faster search, smaller RAM. MPNet: better recall on tricky queries, more expensive everywhere. In production you'd A/B them on your labeled eval set — which is exactly what `04_evaluate.py` gives you.

In [ ]:
# WARNING: first run downloads ~420 MB. Only run once.
# from sentence_transformers import SentenceTransformer
# mpnet = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
# print("embedding dim:", mpnet.get_sentence_embedding_dimension())

## 5. Your notes

As you experiment, jot findings here. Future-you (and interviewers reading this repo) will thank you.

| Date | Change | What I saw | Why I think that happened |
|------|--------|-----------|--------------------------|
| YYYY-MM-DD | e.g. chunk_size 32→64 | recall@3 went from 0.78 → 0.71 | larger chunks average multiple concepts, embedding loses sharpness on narrow queries |
| | | | |
